In [1]:
import os
import json
import random
import pandas as pd
import chromadb
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables (OPENAI_API_KEY)
load_dotenv()

True

In [2]:
# Initialize OpenAI Client
client_openai = OpenAI()

def get_completion(prompt, model="gpt-4o-mini", max_tokens=150, temperature=0):
    """
    Standard completion function using OpenAI Chat interface.
    Defaulting to gpt-3.5-turbo for cost efficiency.
    """
    try:
        response = client_openai.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [3]:
# Testing the connection
print(get_completion("Hello, my name is", max_tokens=10))

Hello! It seems like your message got cut off


In [5]:
# Define the OpenAI model to use
openai_model = "gpt-4o-mini"

print(get_completion("Hello, my name is", model=openai_model))

Hello! It looks like your message got cut off. What’s your name?


In [7]:
prompt = """
Given the following wedding guest data, write a very short 3-sentences thank you letter:

{
  "name": "John Doe",
  "relationship": "Bride's cousin",
  "hometown": "New York, NY",
  "fun_fact": "Climbed Mount Everest in 2020",
  "attending_with": "Sophia Smith",
  "bride_groom_name": "Tom and Mary"
}

Use only the data provided in the JSON object above.
The senders of the letter are the bride and groom, Tom and Mary.
"""

print(get_completion(prompt, model=openai_model, max_tokens=150))

Dear John,  

Thank you for celebrating our special day with us! We loved having you and Sophia there, and your incredible achievement of climbing Mount Everest in 2020 truly inspired us. We hope you enjoyed the wedding as much as we enjoyed having you with us!  

Warm regards,  
Tom and Mary


In [ ]:
# Data Loading
ml_papers = pd.read_csv("dataset/ml-potw-10232023.csv", header=0)
# Remove rows with empty titles or descriptions
ml_papers = ml_papers.dropna(subset=["Title", "Description"])
print(f"Loaded {len(ml_papers)} papers.")

ml_papers.head(5)

Loaded 415 papers.


,Title,Description,PaperURL,TweetURL,Abstract
0,Llemma,an LLM for mathematics which is based on conti...,https://arxiv.org/abs/2310.10631,https://x.com/zhangir_azerbay/status/171409802...,"We present Llemma, a large language model for ..."
1,LLMs for Software Engineering,a comprehensive survey of LLMs for software en...,https://arxiv.org/abs/2310.03533,https://x.com/omarsar0/status/1713940983199506...,This paper provides a survey of the emerging a...
2,Self-RAG,presents a new retrieval-augmented framework t...,https://arxiv.org/abs/2310.11511,https://x.com/AkariAsai/status/171511027707796...,"Despite their remarkable capabilities, large l..."
3,Retrieval-Augmentation for Long-form Question ...,explores retrieval-augmented language models o...,https://arxiv.org/abs/2310.12150,https://x.com/omarsar0/status/1714986431859282...,We present a study of retrieval-augmented lang...
4,GenBench,presents a framework for characterizing and un...,https://www.nature.com/articles/s42256-023-007...,https://x.com/AIatMeta/status/1715041427283902...,NaN


In [ ]:
# Data Transformation
ml_papers_dict = ml_papers.to_dict(orient="records")

if ml_papers_dict:
    print(ml_papers_dict[0])

{'Title': 'Llemma', 'Description': 'an LLM for mathematics which is based on continued pretraining from Code Llama on the Proof-Pile-2 dataset; the dataset involves scientific paper, web data containing mathematics, and mathematical code; Llemma outperforms open base models and the unreleased Minerva on the MATH benchmark; the model is released, including dataset and code to replicate experiments.', 'PaperURL': 'https://arxiv.org/abs/2310.10631', 'TweetURL': 'https://x.com/zhangir_azerbay/status/1714098025956864031?s=20', 'Abstract': 'We present Llemma, a large language model for mathematics. We continue pretraining Code Llama on the Proof-Pile-2, a mixture of scientific papers, web data containing mathematics, and mathematical code, yielding Llemma. On the MATH benchmark Llemma outperforms all known open base models, as well as the unreleased Minerva model suite on an equi-parameter basis. Moreover, Llemma is capable of tool use and formal theorem proving without any further finetunin

In [13]:
# ChromaDB and Embedding Setup

from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer

# Local embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

class MyEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        batch_embeddings = embedding_model.encode(input)
        return batch_embeddings.tolist()

embed_fn = MyEmbeddingFunction()

# Initialize ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path="./chromadb")

# Create or get collection
collection = chroma_client.get_or_create_collection(
    name="ml-papers-nov-2023"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Michael2\AppData\Local\Temp\ipykernel_29748\3963166748.py:14: DeprecationWarning: The class MyEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = MyEmbeddingFunction()


In [14]:
# Vector Indexing
batch_size = 50

for i in tqdm(range(0, len(ml_papers_dict), batch_size)):
    i_end = min(i + batch_size, len(ml_papers_dict))
    batch = ml_papers_dict[i : i + batch_size]

    batch_titles = [str(paper["Title"]) if str(paper["Title"]) != "" else "No Title" for paper in batch]
    
    # Generate unique IDs based on title and a random seed
    batch_ids = [f"id_{i}_{j}" for j, paper in enumerate(batch)]
    
    batch_metadata = [
        dict(url=str(paper.get("PaperURL", "")), abstract=str(paper.get("Abstract", "")))
        for paper in batch
    ]

    # Generate embeddings locally
    batch_embeddings = embedding_model.encode(batch_titles)

    # Upsert to Chroma
    collection.upsert(
        ids=batch_ids,
        metadatas=batch_metadata,
        documents=batch_titles,
        embeddings=batch_embeddings.tolist(),
    )

  0%|          | 0/9 [00:00<?, ?it/s]

In [16]:
# Querying the Database

retriever_results = collection.query(
    query_texts=["Software Engineering"],
    n_results=2,
)

print("Search results for 'Software Engineering':")
print(retriever_results["documents"])

C:\Users\Michael2\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:37<00:00, 2.24MiB/s]


Search results for 'Software Engineering':
[['LLMs for Software Engineering', 'Communicative Agents for Software Development']]


In [17]:
# RAG Generation with OpenAI
# User query
user_query = "S3Eval: A Synthetic, Scalable, Systematic Evaluation Suite for Large Language Models"

# Retrieve similar titles
results = collection.query(
    query_texts=[user_query],
    n_results=10,
)

short_titles = '\n'.join(results['documents'][0])

# Prompt optimized for OpenAI (no [INST] tags)
prompt_template = f'''
Your main task is to generate 5 SUGGESTED_TITLES based on the PAPER_TITLE provided below.

You should mimic a similar style and length as the provided SHORT_TITLES. 
DO NOT include titles from SHORT_TITLES in your output. Only generate new variations of the PAPER_TITLE.

PAPER_TITLE: {user_query}

SHORT_TITLES: 
{short_titles}

SUGGESTED_TITLES:
'''

# Execute completion
suggested_titles = get_completion(prompt_template, model=openai_model, max_tokens=500)

print("-" * 30)
print("Model Suggestions:")
print(suggested_titles)
print("-" * 30)
print("Prompt Template Used:")
print(prompt_template)

------------------------------
Model Suggestions:
1. S3Eval: A Comprehensive Framework for Evaluating Large Language Model Performance  
2. S3Eval: Systematic Assessment of Scalability in Language Models  
3. S3Eval: A Synthetic Benchmark for Large Language Model Evaluation  
4. S3Eval: Scalable Evaluation Techniques for Advanced Language Models  
5. S3Eval: A Systematic Approach to Testing Large Language Model Capabilities  
------------------------------
Prompt Template Used:

Your main task is to generate 5 SUGGESTED_TITLES based on the PAPER_TITLE provided below.

You should mimic a similar style and length as the provided SHORT_TITLES. 
DO NOT include titles from SHORT_TITLES in your output. Only generate new variations of the PAPER_TITLE.

PAPER_TITLE: S3Eval: A Synthetic, Scalable, Systematic Evaluation Suite for Large Language Models

SHORT_TITLES: 
Pythia: A Suite for Analyzing Large Language Models Across Training and Scaling
ChemCrow: Augmenting large-language models with ch